#  UFC Fighters — Análisis Exploratorio y Visualizaciones Interactivas

**Asignatura:** Programación para la Ciencia de Datos — Evaluación Parcial 3
**Dataset:** UFC Fighters Stats (4.496 peleadores, estadísticas físicas y de combate)
**Pregunta de análisis:** ¿Qué características físicas y de estilo distinguen a los peleadores de UFC más exitosos?

**Visualizaciones interactivas (Plotly):** Gráfico de barras · Gráfico de líneas · Gráfico de dispersión

---

## 1.  Importar Librerías

In [ ]:
# ── Manipulación de datos ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualización interactiva (Plotly) ─────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go

# Configuración para que las tablas se vean ordenadas
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)

print('Librerías importadas correctamente')

## 2.  Cargar el Dataset

Usamos el archivo `ufc_fighters_stats_merged.csv`, que combina los datos físicos de cada peleador
(altura, alcance, postura) con sus estadísticas de combate (golpes por minuto, precisión, victorias por KO, etc.).

### Subir el archivo CSV (solo en Google Colab)

Ejecuta la celda de abajo y selecciona el archivo **ufc_fighters_stats_merged.csv** desde tu computador.
Si no estás en Colab, puedes saltarte esta celda y dejar el CSV en la misma carpeta del notebook.

In [ ]:
# ── Subir el CSV en Google Colab ───────────────────────────────────────────────
# Al ejecutar, aparece un botón para seleccionar el archivo desde tu PC.
try:
    from google.colab import files
    subido = files.upload()   # selecciona ufc_fighters_stats_merged.csv
except ImportError:
    print('No estás en Colab — asegúrate de tener el CSV en la misma carpeta.')

In [ ]:
# Cargar el CSV — ajusta la ruta si es necesario
df = pd.read_csv('ufc_fighters_stats_merged.csv')

print(f' Shape: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'\n Columnas:')
print(list(df.columns))

In [ ]:
# Vista rápida de las primeras filas
df.head()

In [ ]:
# Estadísticas descriptivas de columnas numéricas
df.describe().round(2)

## 3.  Exploración de Calidad de los Datos

Antes de limpiar, revisamos **cuántos nulos** tiene cada columna. Esto define qué columnas podemos usar
y cuáles hay que descartar.

In [ ]:
# ── Porcentaje de valores nulos por columna ────────────────────────────────────
nulos = (df.isnull().mean() * 100).round(1).sort_values(ascending=False)
print('Porcentaje de nulos por columna:\n')
print(nulos.to_string())

**Observación importante:**

- Tres columnas están casi vacías: `str_defense_pct` (99.6% nulos), `td_defense_pct` (87.7%) y `takedowns_landed` (79.6%).
  **No se pueden usar** — imputarlas significaría inventar el 80-99% de los datos, lo cual no es válido.
- Las estadísticas de golpeo (`str_landed_per_min`, `striking_accuracy_pct`, etc.) tienen ~33% de nulos.
  Esto es manejable: son peleadores antiguos sin registro detallado de golpes. Los trataremos según cada análisis.
- Los datos básicos (`wins`, `losses`, `height_cm`, `weight`) están casi completos.

## 4.  Limpieza y Preparación de los Datos

Aplicamos cuatro pasos de limpieza, justificando cada decisión.

In [ ]:
# ── Paso 1: Descartar columnas con demasiados nulos (>85%) ─────────────────────
# No se pueden imputar de forma honesta, así que las eliminamos
columnas_inservibles = ['str_defense_pct', 'td_defense_pct', 'takedowns_landed']
df = df.drop(columns=columnas_inservibles)
print(f'Columnas descartadas por exceso de nulos: {columnas_inservibles}')

# ── Paso 2: Crear columna de nombre completo ───────────────────────────────────
df['name'] = (df['first'].fillna('') + ' ' + df['last'].fillna('')).str.strip()

# ── Paso 3: Crear variables derivadas útiles ───────────────────────────────────
# Total de peleas y tasa de victorias (win rate)
df['total_fights'] = df['wins'] + df['losses'] + df['draws']

# Evitamos dividir por cero: solo calculamos win_rate si hay al menos 1 pelea
df['win_rate'] = np.where(df['total_fights'] > 0,
                          df['wins'] / df['total_fights'],
                          np.nan)

print('Variables creadas: name, total_fights, win_rate')

In [ ]:
# ── Paso 4: Agrupar posturas (stance) poco frecuentes ──────────────────────────
# 'Open Stance' (7) y 'Sideways' (3) son muy raras → las agrupamos en 'Other'
df['stance'] = df['stance'].replace({'Open Stance': 'Other', 'Sideways': 'Other'})

print('Distribución de postura (stance) tras limpieza:')
print(df['stance'].value_counts(dropna=False))

# ── Tratar el peso físicamente implausible (770 lb ≈ 349 kg) ───────────────────
# El valor de 770 lb es sospechoso (probable error de captura). No lo borramos del
# dataset original, pero creamos un subconjunto acotado a un rango realista (≤ 300 lb
# cubre todas las divisiones oficiales de UFC) para los análisis de peso.
df_peso = df[df['weight'] <= 300].copy()
print(f'Peleadores en rango de peso realista (≤300 lb): {len(df_peso)} de {len(df)}')
print(f'Excluidos por peso atípico: {len(df) - len(df_peso)}')

**Sobre los pesos muy altos:** El dataset registra pesos en libras de hasta 770 lb (~349 kg), un valor físicamente implausible y probablemente un error de captura. No alteramos los datos originales, pero para los análisis de peso usamos un subconjunto acotado a ≤ 300 lb (rango que cubre todas las divisiones oficiales de UFC), evitando que ese valor atípico distorsione los gráficos.

In [ ]:
# ── Verificación final de la limpieza ──────────────────────────────────────────
print(f'Dataset limpio: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'Peleadores con al menos 1 pelea registrada: {(df["total_fights"] > 0).sum()}')
print(f'Peleadores con stats de golpeo completas: {df["str_landed_per_min"].notna().sum()}')

---
## 5.  Mini Dashboard — Resultados Principales

A continuación presentamos las **tres visualizaciones interactivas** principales construidas con **Plotly**.
Cada una responde a una parte de nuestra pregunta y se acompaña de su interpretación.

| # | Tipo de gráfico | Qué responde |
|---|---|---|
| 1 | Barras | ¿Qué postura (stance) tiene mejor tasa de victorias? |
| 2 | Líneas | ¿Cómo se relaciona el alcance con el peso del peleador? |
| 3 | Dispersión | ¿Volumen vs. precisión de golpeo distingue a los peleadores? |
| 4 | Barras | ¿Cómo ganan los peleadores: KO, sumisión o decisión? |
| 5 | Dispersión | ¿Golpear más implica absorber más castigo? |

### 5.1  Gráfico de Barras — Tasa de Victorias por Postura

Comparamos la **tasa de victorias promedio** según la postura del peleador.
Filtramos a peleadores con **al menos 5 peleas** para que el promedio sea confiable
(un peleador con 1 pelea ganada tendría 100% de win rate, lo cual no es representativo).

In [ ]:
# ── Preparar datos: win rate medio por stance (peleadores con >=5 peleas) ──────
exp = df[(df['total_fights'] >= 5) & (df['stance'].notna())].copy()

win_por_stance = (exp.groupby('stance')['win_rate']
                     .mean()
                     .reset_index()
                     .sort_values('win_rate', ascending=False))
win_por_stance['win_rate_pct'] = (win_por_stance['win_rate'] * 100).round(1)

# ── Gráfico de barras interactivo con Plotly Express ───────────────────────────
fig1 = px.bar(
    win_por_stance,
    x='stance',
    y='win_rate_pct',
    color='stance',
    text='win_rate_pct',
    title='Tasa de Victorias Promedio según Postura (peleadores con ≥5 peleas)',
    labels={'stance': 'Postura', 'win_rate_pct': 'Tasa de victorias (%)'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig1.update_traces(texttemplate='%{text}%', textposition='outside')
fig1.update_layout(showlegend=False, yaxis_range=[0, 85])
fig1.show()

** Interpretación:**
La postura **Switch** (peleadores que cambian de guardia) muestra la tasa de victorias más alta (~74%),
seguida de cerca por **Southpaw** y **Orthodox** (~70%). La diferencia es moderada pero consistente:
la versatilidad de cambiar de postura parece dar una pequeña ventaja, posiblemente porque vuelve al peleador
más impredecible. Sin embargo, la mayoría de peleadores son Orthodox, así que la muestra de Switch es menor.

### 5.2  Gráfico de Líneas — Alcance Promedio según Peso

Analizamos cómo cambia el **alcance promedio** (reach) a medida que aumenta el **peso** del peleador.
Agrupamos el peso en rangos (divisiones aproximadas) para ver la tendencia con claridad.

In [ ]:
# ── Preparar datos: alcance medio por rango de peso ────────────────────────────
# Usamos df_peso (rango realista <=300 lb) y filtramos a quienes tienen reach registrado
peso_reach = df_peso[df_peso['reach'].notna()].copy()

# Crear rangos de peso de 20 en 20 libras
peso_reach['rango_peso'] = pd.cut(peso_reach['weight'],
                                  bins=range(100, 320, 20))
linea = (peso_reach.groupby('rango_peso', observed=True)['reach']
                   .mean()
                   .reset_index())
# Usar el punto medio del rango como eje X numérico
linea['peso_medio'] = linea['rango_peso'].apply(lambda x: x.mid).astype(float)

# ── Gráfico de líneas interactivo ──────────────────────────────────────────────
fig2 = px.line(
    linea,
    x='peso_medio',
    y='reach',
    markers=True,
    title='Alcance Promedio (reach) según el Peso del Peleador',
    labels={'peso_medio': 'Peso (libras)', 'reach': 'Alcance promedio (pulgadas)'}
)
fig2.update_traces(line=dict(width=3), marker=dict(size=9))
fig2.show()

** Interpretación:**
Existe una relación **creciente y casi lineal**: a mayor peso, mayor alcance promedio. Esto es esperable
físicamente (peleadores más pesados tienden a ser más altos y con brazos más largos), pero el gráfico lo
**cuantifica**: el alcance pasa de ~68 pulgadas en pesos ligeros a más de 75 pulgadas en pesos pesados.
La tendencia se mantiene estable, lo que confirma que el peso es un buen predictor del alcance.

### 5.3  Gráfico de Dispersión — Volumen vs. Precisión de Golpeo

Este es el gráfico central. Cruzamos dos métricas de golpeo:
- **Eje X:** golpes significativos conectados por minuto (volumen de ataque)
- **Eje Y:** precisión de golpeo (% de golpes que aciertan)

Coloreamos por postura y el tamaño representa el total de peleas, para ver si hay un perfil dominante.

In [ ]:
# ── Preparar datos: solo peleadores con stats de golpeo y algo de experiencia ──
scatter_df = df[
    df['str_landed_per_min'].notna() &
    df['striking_accuracy_pct'].notna() &
    df['stance'].notna() &
    (df['total_fights'] >= 3)
].copy()

# Filtrar valores extremos poco representativos para que el gráfico sea legible
scatter_df = scatter_df[
    (scatter_df['str_landed_per_min'] <= 12) &
    (scatter_df['striking_accuracy_pct'] <= 100)
]

# ── Gráfico de dispersión interactivo (estilo del ejemplo del curso) ───────────
fig3 = px.scatter(
    scatter_df,
    x='str_landed_per_min',
    y='striking_accuracy_pct',
    color='stance',
    size='total_fights',
    hover_name='name',
    hover_data={'wins': True, 'losses': True, 'total_fights': True},
    title='Volumen vs. Precisión de Golpeo por Postura',
    labels={
        'str_landed_per_min': 'Golpes conectados por minuto',
        'striking_accuracy_pct': 'Precisión de golpeo (%)',
        'stance': 'Postura'
    },
    opacity=0.6,
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig3.show()

** Interpretación:**
La nube de puntos muestra que **no hay correlación fuerte** entre volumen y precisión: hay peleadores que
golpean mucho con baja precisión y otros que golpean poco pero certero. La mayoría se concentra entre
**2 y 5 golpes/min** con **40-55% de precisión**. Las posturas se mezclan sin formar grupos separados, lo
que sugiere que el estilo de golpeo depende más del peleador individual que de su postura. Pasa el cursor
sobre los puntos grandes (más peleas) para identificar a los veteranos del deporte.

### 5.4  Gráfico de Barras — ¿Cómo Ganan los Peleadores?

Más allá de cuánto ganan, nos interesa **cómo** ganan. Comparamos el porcentaje promedio de victorias
logradas por **KO/TKO** (nocaut), **sumisión** (rendición) y **decisión** (por puntos de los jueces).

In [ ]:
# ── Promedio de cada tipo de victoria (peleadores con desglose registrado) ─────
tipos = df.dropna(subset=['ko_win_percentage', 'sub_win_percentage', 'dec_win_percentage'])

medias = pd.DataFrame({
    'Tipo de victoria': ['KO / TKO', 'Sumisión', 'Decisión'],
    'Porcentaje promedio': [
        tipos['ko_win_percentage'].mean(),
        tipos['sub_win_percentage'].mean(),
        tipos['dec_win_percentage'].mean()
    ]
})
medias['Porcentaje promedio'] = medias['Porcentaje promedio'].round(1)

# ── Gráfico de barras interactivo ──────────────────────────────────────────────
fig4 = px.bar(
    medias,
    x='Tipo de victoria',
    y='Porcentaje promedio',
    color='Tipo de victoria',
    text='Porcentaje promedio',
    title='Cómo Ganan los Peleadores de UFC (promedio por tipo de victoria)',
    labels={'Porcentaje promedio': 'Porcentaje promedio de victorias (%)'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig4.update_traces(texttemplate='%{text}%', textposition='outside')
fig4.update_layout(showlegend=False)
fig4.show()

 **Interpretación:**
El **KO/TKO es la forma de victoria más frecuente** (~21% promedio), por encima de sumisión (~15%) y
decisión (~15%). Esto refleja que el golpeo de poder es el camino más común a la victoria en UFC.
Que sumisión y decisión estén casi empatadas sugiere dos perfiles distintos pero igual de presentes:
los especialistas en suelo que buscan la rendición, y los peleadores que llevan el combate a los puntos.

### 5.5  Gráfico de Dispersión — Golpes Conectados vs. Absorbidos

¿Los peleadores que más golpean son también los que más castigo reciben? Cruzamos los golpes que un
peleador **conecta** por minuto contra los que **absorbe** por minuto, para ver si existe un perfil
"agresivo pero vulnerable" o si hay peleadores que golpean mucho sin recibir.

In [ ]:
# ── Preparar datos: filtrar valores extremos para legibilidad ─────────────────
golpeo = df.dropna(subset=['str_landed_per_min', 'str_absorbed_per_min', 'stance']).copy()
golpeo = golpeo[
    (golpeo['str_landed_per_min'] <= 12) &
    (golpeo['str_absorbed_per_min'] <= 12) &
    (df['total_fights'] >= 3)
]

# ── Gráfico de dispersión interactivo ──────────────────────────────────────────
fig5 = px.scatter(
    golpeo,
    x='str_landed_per_min',
    y='str_absorbed_per_min',
    color='stance',
    hover_name='name',
    hover_data={'wins': True, 'losses': True},
    title='Golpes Conectados vs. Absorbidos por Minuto',
    labels={
        'str_landed_per_min': 'Golpes conectados por minuto',
        'str_absorbed_per_min': 'Golpes absorbidos por minuto',
        'stance': 'Postura'
    },
    opacity=0.5,
    color_discrete_sequence=px.colors.qualitative.Bold
)
# Línea de referencia: por encima = recibe más de lo que da
fig5.add_shape(type='line', x0=0, y0=0, x1=12, y1=12,
               line=dict(dash='dash', color='gray'))
fig5.show()

 **Interpretación:**
**No hay una relación fuerte** entre golpear y ser golpeado (correlación ≈ 0.20): golpear mucho NO implica
recibir mucho. La línea punteada marca el equilibrio — los puntos **por debajo** son peleadores que conectan
más de lo que absorben (perfil eficiente/defensivo), y los de **arriba** reciben más castigo del que reparten.
La mayoría se concentra en la zona baja (2-5 golpes), lo que indica que el peleador promedio mantiene un
intercambio controlado. Los puntos aislados arriba son peleadores de estilo arriesgado.

---
## 6.  Resumen del Dashboard

Combinamos los hallazgos en una tabla de resultados clave para cerrar el análisis.

In [ ]:
# ── Tabla resumen de hallazgos ─────────────────────────────────────────────────
resumen = pd.DataFrame({
    'Hallazgo': [
        'Postura con mejor tasa de victorias',
        'Tasa de victorias general (Orthodox)',
        'Relación peso-alcance',
        'Precisión de golpeo más común',
        'Volumen de golpeo más común'
    ],
    'Resultado': [
        'Switch (~74%)',
        '~70%',
        'Creciente (a más peso, más alcance)',
        '40% - 55%',
        '2 - 5 golpes por minuto'
    ]
})
resumen

In [ ]:
# ── Métricas globales del dataset (tarjetas tipo KPI) ──────────────────────────
print('═══════════ MÉTRICAS GLOBALES DEL DATASET ═══════════')
print(f'  Total de peleadores analizados : {len(df):,}')
print(f'  Peleadores con stats completas : {df["str_landed_per_min"].notna().sum():,}')
print(f'  Altura promedio                : {df["height_cm"].mean():.1f} cm')
print(f'  Tasa de victorias promedio     : {df["win_rate"].mean()*100:.1f}%')
print(f'  Postura más común              : {df["stance"].mode()[0]}')
print('═════════════════════════════════════════════════════')

---
## 7.  Conclusiones

**Respuesta a la pregunta:** *¿Qué características distinguen a los peleadores de UFC más exitosos?*

1. **La postura importa poco, pero la versatilidad ayuda.** Los peleadores *Switch* ganan algo más seguido (~74%),
   aunque la diferencia con Orthodox/Southpaw (~70%) es moderada.

2. **El peso predice el alcance de forma clara.** La relación es creciente y estable: es un patrón físico
   sólido que se confirma con el gráfico de líneas.

3. **No existe un "estilo de golpeo ganador" único.** El gráfico de dispersión muestra que volumen y precisión
   no están correlacionados y que las posturas se mezclan. El éxito depende más del peleador individual que de
   una sola métrica.

**Sobre el trabajo de datos:** El mayor desafío fue la **calidad de los datos** — descartamos 3 columnas
inservibles por exceso de nulos y filtramos rangos extremos, todo documentado para no inventar información.
Esto demuestra que un buen análisis empieza por una limpieza honesta y justificada.

**Herramientas usadas:** pandas (manipulación y limpieza) · Plotly Express (visualizaciones interactivas) · Git/GitHub (control de versiones).